# admin

> Item management CLI — `python -m habitrack.admin` (see `DESIGN.md` §5). Called by the `habitrack-items` skill over ssh; no UI in the app.

In [ ]:
#| default_exp admin

In [ ]:
#| export
import argparse
import json
import shutil
import sys
from datetime import date, datetime
from pathlib import Path

from habitrack.app import TZ
from habitrack.core import (add_item, display_name, is_archived, load_full, ordered_items,
                            rate_30d, rename_item, save_full, set_archived)

In [ ]:
#| hide
import io
import tempfile
from contextlib import redirect_stdout, redirect_stderr

def raises(exc, fn, *args):
    "Assert that `fn(*args)` raises `exc`."
    try: fn(*args)
    except exc: return
    raise AssertionError(f"{exc.__name__} not raised")

def sample(d):
    "Temp ledger with one active and one archived item."
    p = Path(d) / "ledger.csv"
    p.write_text("date,読書,~運動\n2026-08-23,True,False\n2026-08-24,True,True\n")
    return p

## `backup`

In [ ]:
#| export
def backup(
    path:Path,  # live ledger CSV
) -> Path:
    "Copy the ledger to a `.bak` sibling (overwritten each time) — the only safety net before a mutation."
    bak = path.with_suffix(".bak")
    shutil.copy2(path, bak)
    return bak

In [ ]:
with tempfile.TemporaryDirectory() as d:
    p = sample(d)
    bak = backup(p)
    assert bak == p.with_suffix(".bak") and bak.read_text() == p.read_text()
    p.write_text("date,x\n")
    assert backup(p).read_text() == "date,x\n"  # overwritten, not appended

## `summary`

In [ ]:
#| export
def summary(
    path:Path,  # live ledger CSV; must exist
    today:date,  # last day of the 30-day rate window
) -> list:
    "One dict per item in display order: `name` (no marker), `archived`, `rate_30d`."
    if not path.exists():
        raise FileNotFoundError(path)
    full = load_full(path, [], today)
    return [{"name": display_name(c), "archived": is_archived(c), "rate_30d": rate_30d(full, c, today)}
            for c in ordered_items(full)]

In [ ]:
with tempfile.TemporaryDirectory() as d:
    p = sample(d)
    t = date(2026, 8, 24)
    assert summary(p, t) == [
        {"name": "読書", "archived": False, "rate_30d": 2 / 30},
        {"name": "運動", "archived": True, "rate_30d": 1 / 30},
    ]
    raises(FileNotFoundError, summary, Path(d) / "missing.csv", t)  # never auto-create from admin

## `mutate`

In [ ]:
#| export
def mutate(
    path:Path,  # live ledger CSV; must exist
    fn,  # pure transform `DataFrame -> DataFrame` from `habitrack.core`
    today:date,  # passed through to `summary`
) -> list:
    "backup -> load -> fn -> atomic save -> re-read check; returns the new summary."
    if not path.exists():
        raise FileNotFoundError(path)
    df = fn(load_full(path, [], today))  # validate before touching disk
    backup(path)
    save_full(df, path)
    if not load_full(path, [], today).equals(df):
        raise RuntimeError(f"re-read mismatch after writing {path}; previous state is in {path.with_suffix('.bak')}")
    return summary(path, today)

In [ ]:
with tempfile.TemporaryDirectory() as d:
    p = sample(d)
    t = date(2026, 8, 24)
    out = mutate(p, lambda df: add_item(df, "新項目"), t)
    assert [i["name"] for i in out] == ["読書", "新項目", "運動"]
    assert p.with_suffix(".bak").read_text().startswith("date,読書,~運動")  # pre-mutation copy
    assert load_full(p, [], t).at[t, "新項目"] == False
    before = p.read_text()
    raises(ValueError, mutate, p, lambda df: add_item(df, "読書"), t)  # validation error -> file untouched
    assert p.read_text() == before

## `main`

In [ ]:
#| export
def main(
    argv:list=None,  # CLI args; defaults to sys.argv[1:]
) -> int:
    "Subcommands: summary | add NAME | archive NAME | unarchive NAME | rename OLD NEW. JSON on stdout, errors on stderr."
    p = argparse.ArgumentParser(prog="python -m habitrack.admin", description=main.__doc__)
    p.add_argument("--data", type=Path, required=True, help="ledger CSV, e.g. /opt/habitrack/app/data/ledger.csv")
    sub = p.add_subparsers(dest="cmd", required=True)
    sub.add_parser("summary")
    for cmd in ["add", "archive", "unarchive"]:
        sub.add_parser(cmd).add_argument("name")
    r = sub.add_parser("rename")
    r.add_argument("old")
    r.add_argument("new")
    a = p.parse_args(argv)
    ops = {
        "add": lambda df: add_item(df, a.name),
        "archive": lambda df: set_archived(df, a.name, True),
        "unarchive": lambda df: set_archived(df, a.name, False),
        "rename": lambda df: rename_item(df, a.old, a.new),
    }
    today = datetime.now(TZ).date()
    try:
        out = summary(a.data, today) if a.cmd == "summary" else mutate(a.data, ops[a.cmd], today)
    except (ValueError, KeyError, FileNotFoundError, RuntimeError) as e:
        print(f"error: {e.args[0] if e.args else e}", file=sys.stderr)
        return 1
    print(json.dumps(out, ensure_ascii=False))
    return 0

In [ ]:
#| export
#| eval: false
if __name__ == "__main__":  # python -m habitrack.admin
    sys.exit(main())

In [ ]:
def run(*args):
    "Call `main` capturing (exit code, stdout, stderr)."
    out, err = io.StringIO(), io.StringIO()
    with redirect_stdout(out), redirect_stderr(err):
        rc = main(list(args))
    return rc, out.getvalue(), err.getvalue()

with tempfile.TemporaryDirectory() as d:
    p = str(sample(d))
    rc, out, err = run("--data", p, "summary")
    assert rc == 0 and err == ""
    assert [i["name"] for i in json.loads(out)] == ["読書", "運動"]

    rc, out, _ = run("--data", p, "add", "新項目")
    assert rc == 0 and [i["name"] for i in json.loads(out)] == ["読書", "新項目", "運動"]
    rc, out, _ = run("--data", p, "archive", "読書")
    assert rc == 0 and [i["name"] for i in json.loads(out)] == ["新項目", "読書", "運動"]
    rc, out, _ = run("--data", p, "unarchive", "運動")
    assert rc == 0 and [i["name"] for i in json.loads(out)] == ["運動", "新項目", "読書"]  # column order kept
    rc, out, _ = run("--data", p, "rename", "運動", "筋トレ")
    assert rc == 0 and [i["name"] for i in json.loads(out)] == ["筋トレ", "新項目", "読書"]

    rc, out, err = run("--data", p, "add", "a/b")  # validation error: exit 1, message on stderr, no JSON
    assert rc == 1 and out == "" and "forbidden" in err
    rc, out, err = run("--data", p, "archive", "無い")
    assert rc == 1 and out == "" and "無い" in err

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()